> **⚠ SUPERSEDED FRAMING (2026-06-27).** This notebook is a point-in-time record; its "5/6 ceiling / 6/6" framing is superseded by the surrogate-to-model **identifiability study** (4 observable params {alpfe, scav_rat, diatomgraz, R_PICPOC}; growth pair unobservable by construction; **R_PICPOC is recoverable** via a real calcite anchor — the differentiable Darwin port was tested and did not help). The surrogate gap is dimensional (the 0-D box homogenizes spatial structure). See `STATUS.md` / `README.md`.

# v3.0 multi-AOI parameter learner ceiling -- arc analysis

This notebook closes out the v3.0 multi-AOI joint-training arc (PRs #44 - #59).
It loads every JSON produced across the per-AOI-DINN, PIC-absolute, and paired POC+PIC
absolute experiments, aggregates them against the PR #57 best-config baseline, and
characterizes the 5/6 ceiling structurally.

**Headline findings**

1. **Baseline aggregate is 7/15 at 5/6 Cal-grade, mean_cal = 3.93.** R_PICPOC mean
   joint = 0.030 (only 0.36 off Carroll = Cal-grade); the previously-claimed
   'R_PICPOC stuck at 0.014' was a per-AOI-DINN 5/6-miss-seed artifact, not the
   baseline state.
2. **The actual binding parameter is `diatomgraz`** (2/15 Cal in baseline; 6 of 7
   5/6-miss seeds drop diatomgraz). R_PICPOC is Cal-grade in 11/15 baseline seeds.
3. **PR #58 per-AOI DINNs + consistency penalty:** falsified -- best lambda (0.1)
   delivers 3/10 at 5/6 (below baseline 47%); shifts the dominant-miss to R_PICPOC.
4. **PR #59 paired POC+PIC absolute anchors:** R_PICPOC mean moves to 0.02-0.03
   in some configs (resembling Carroll), but iron-pair Cal collapses to 0/5 seeds
   in every anchored configuration -- light or heavy paired weights both break
   alpfe/scav_rat.
5. **Parameter conservation:** no anchored configuration beats baseline on aggregate.
   Each parameter has bounded support from the observations; pushing one up pulls
   another down.
6. **Implication for 6/6:** the laptop-tractable break path is action #2 from the
   original v3.0 next-action stack -- a POSi (biogenic silica) observation that
   targets `diatomgraz` directly, the actual binding parameter.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

def find_repo_root() -> Path:
    # Walk up from cwd until we find the src/darwindiff marker; robust to
    # whatever cwd the kernel was started with.
    p = Path.cwd().resolve()
    for d in [p, *p.parents]:
        if (d / 'src' / 'darwindiff').is_dir():
            return d
    raise RuntimeError(f'repo root not found from {p}')

ROOT = find_repo_root()
ARC = ROOT / 'docs' / 'findings' / 'v3.0_arc'
SCRIPTS = ROOT / 'scripts'
print(f'ROOT={ROOT}')

# Re-run the data engine to ensure the CSVs + plots reflect the latest JSONs on disk.
import subprocess
subprocess.run([sys.executable, str(SCRIPTS / 'v3.0_arc_analysis.py')], check=True, cwd=ROOT)

summary = pd.read_csv(ARC / 'summary.csv')
params = pd.read_csv(ARC / 'param_bands.csv')
obs = pd.read_csv(ARC / 'obs_pic_poc_per_aoi.csv')
print(f'summary: {len(summary)} rows; params: {len(params)} rows')

## 1. Per-config aggregate

Each row is one configuration. `R_PICPOC_off` is mean absolute relative offset
from Carroll (Cal-grade threshold: <= 0.50). Only **baseline_pr57** has alpfe
Cal rate > 0; every anchored family collapses to alpfe_cal_rate = 0.

In [ ]:
cols = ['family', 'config', 'n', 'n_at_5', 'mean_cal', 'total_exc',
        'R_PICPOC_mean', 'R_PICPOC_off',
        'alpfe_cal_rate', 'scav_rat_cal_rate', 'R_PICPOC_cal_rate']
summary[cols].round(3)

## 2. The actual binding parameter -- diatomgraz, not R_PICPOC

Aggregate Cal hit rate per parameter, by config. In the baseline, all six params
have decent Cal rates EXCEPT diatomgraz (2/15 = 13%). 6 of 7 baseline 5/6-miss
seeds specifically drop diatomgraz.

PR #58's per-AOI DINN runs SHIFTED the binding param to R_PICPOC by disturbing the
baseline equilibrium; PR #59's paired anchors shifted it to alpfe/scav_rat. But in
the baseline, **diatomgraz is the parameter the observations can't reach**.

In [ ]:
pivot = params.pivot_table(index='param', columns='config', values='cal_hits',
                            aggfunc='first').reindex(
    ['alpfe', 'scav_rat', 'Smallgrow', 'Biggrow', 'diatomgraz', 'R_PICPOC'])
show_cols = ['baseline', 'picabsW1.0', 'poc1.0_pic1.0', 'poc0.5_pic1.0']
show_cols = [c for c in show_cols if c in pivot.columns]
pivot[show_cols]

## 3. The R_PICPOC vs iron-pair trade-off (the visual)

X = fraction of seeds with `alpfe` at Cal or better. Y = fraction with `R_PICPOC`
at Cal. Diagonal is the 'win-win' line. Baseline (gray) is the only point in the
upper-right. Every anchored configuration collapses to alpfe = 0.

In [ ]:
display(Image(str(ARC / 'tradeoff_iron_vs_rpicpoc.png')))

## 4. Per-AOI R_PICPOC -- the anchors work as designed per-cell

Each point is `eqpac R_PICPOC mean` vs `natlsubpolar R_PICPOC mean` for one
configuration. Dotted lines show observed PIC/POC ratios per AOI from the cached
Darwin v05 targets.

Baseline (gray) lands near Carroll for both AOIs (shared MLP -> single value across
cells). Paired-anchor configs (green / purple) land near the observed ratios per
AOI -- eqpac ~0.03, natl ~0.6-0.7 -- matching theory: paired anchors force
`R_PICPOC = obs(PIC) / obs(POC)` per cell. The mechanism works; the cost is iron
pair.

In [ ]:
display(Image(str(ARC / 'per_aoi_r_picpoc.png')))
obs.round(4)

## 5. R_PICPOC joint-mean distribution across all 17 configurations

Dashed line: Carroll = 0.04245. Dotted line: Cal-grade threshold (50% off).
Baseline lands ~70% of Carroll (R_PICPOC=0.03, Cal-grade). PIC-alone (red) at
~0.012. Paired heavy (green) and paired light (purple) at ~0.02 -- closer to
Carroll than PIC-alone but still below baseline. **No anchored config beats
baseline on R_PICPOC alone**; the one that comes closest (poc1.0_pic3.0 at 0.023)
does so at full iron-pair collapse.

In [ ]:
display(Image(str(ARC / 'r_picpoc_distribution.png')))

## 6. Parameter conservation -- a structural diagnosis

Across 17 distinct configurations spanning architectural (per-AOI DINNs) and
observational (absolute anchors) interventions, **no configuration beats baseline
on aggregate `mean_cal`**. The pattern is conservation: when an intervention
shifts one parameter toward Carroll, another shifts away.

| Family | Dominant 5/6 miss | Mechanism |
|---|---|---|
| Baseline (PR #57) | **diatomgraz** | Chl1 z-score under-constrains diatom-specific growth |
| Per-AOI DINN (PR #58) | **R_PICPOC** | Removing shared-MLP regularization unbinds carbonate fit |
| PIC alone (PR #59) | **alpfe + scav_rat** | Magnitude anchor on PIC competes with iron budget |
| Paired POC+PIC (PR #59) | **alpfe + scav_rat** | Both anchors disturb iron budget |

The observations provide ~5 effective constraints on 6 parameters. Any 5 can be
jointly identified at Cal-grade; the 6th absorbs the residual bias. Which parameter
is the 'residual sink' depends on the loss weighting.

**6/6 simultaneously requires breaking the conservation:**

- **Add a new independent observation** -- POSi (biogenic silica) constrains
  diatom dynamics directly. ~1280 finite GEOTRACES values; requires extending the
  15-tracer state with a POSi component. **The actionable laptop next step.**
- Reduce parameter count -- fix some Carroll-6 entries at published values.
- Add a 3rd AOI (Southern Ocean Pacific sector).
- Cluster work -- full LLC270 native resolution + multi-basin training.

## 7. v3.0 close-out

**Shipped (5/11 - 5/18, 14 PRs merged + 2 in draft):** multi-AOI joint training,
AOI ID env channel, GEOTRACES POC absolute loss, DINN capacity sweep, per-AOI
weights, dual cellweighted/aoiweighted recovery, CHL1_W_EXTRA, per-AOI DINNs +
consistency penalty, PIC absolute, POC absolute -- all gated and documented.

**Ruled OUT:**
- Shared-MLP architectural ceiling (PR #58 falsified).
- Single-anchor magnitude fix for R_PICPOC (PR #59 sweep 1).
- Paired-anchor magnitude fix without iron-pair disturbance (PR #59 sweeps 2+3).

**Ruled IN:**
- 5/6 ceiling is observational, not architectural.
- The binding parameter at baseline is **diatomgraz**.
- Parameter conservation: ~5 effective observational constraints on 6 params;
  no architectural or weight-tuning trick beats baseline aggregate.

**Recommended next session:** implement POSi loss + biogenic silica state
extension. Diatom-specific observable, directly attacks `diatomgraz` without
competing with the iron-pair budget.